# Informe Técnico: Sistema de Recuperación de Información Multimodal (RAG)

**Asignatura:** Recuperación de Información  
**Proyecto:** Multimodal Information Retrieval System 

**Nombres:** Mateo Coronado, Belén Raura, Edison Quishpe  
**Tecnologías principales:** CLIP (ViT-B-32), ChromaDB, Cross-Encoder, Gemini API, Streamlit  

**Enlace Repositorio:** https://github.com/LAINE30/Multimodal-Information-Retrieval-System.git

## 1. Descripción del Corpus Utilizado

El corpus se construyó a partir del dataset público **Amazon Reviews 2023** (McAuley-Lab, Hugging Face), descargado en modo streaming para eficiencia de memoria.

| Característica | Valor |
| :--- | :--- |
| **Total de documentos** | 700 productos |
| **Categorías** | 10 |
| **Productos por categoría** | ~50 (con filtro de calidad) |
| **Modalidades** | Texto + Imagen |
| **Idioma** | Inglés |
| **Almacenamiento** | `data/processed/corpus.json` + imágenes locales en `data/raw/images/` |

**Categorías incluidas:** Musical Instruments, Video Games, Pet Supplies, Camera & Photo, Electronics, Sports & Outdoors, Toys & Games, Home & Kitchen, Cell Phones & Accessories, Office Products.

**Estructura de cada documento:**
- **id:** Identificador único (`{categoría}_{ASIN}`).
- **title:** Nombre del producto.
- **text:** Concatenación de título + descripción + características del producto.
- **image_url / local_image_path:** URL original y ruta local de la imagen representativa del producto.
- **category:** Categoría de producto.

**Preprocesamiento aplicado:**
- Filtrado de productos sin título, sin imagen o con texto inferior a 20 caracteres.
- Conversión de imágenes a formato JPEG/RGB para uniformidad.
- Eliminación de duplicados por ID.
- Persistencia incremental (si se interrumpe la descarga, se retoma sin duplicar).

## 2. Arquitectura General del Sistema

El sistema sigue una arquitectura **RAG (Retrieval-Augmented Generation)** multimodal con las siguientes capas:

```
┌─────────────────────────────────────────────────────────────┐
│                    INTERFAZ (Streamlit)                     │
│         Texto  ──┐                                         │
│         Voz   ───┼──►  Pipeline de Recuperación + RAG      │
│         Imagen ──┘                                         │
├─────────────────────────────────────────────────────────────┤
│  CAPA DE RECUPERACIÓN                                      │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────────┐  │
│  │ Query        │  │ CLIP         │  │ Cross-Encoder    │  │
│  │ Expansion    │──│ Bi-Encoder   │──│ Re-Ranking       │  │
│  │ (Gemini)     │  │ (ViT-B-32)   │  │ (ms-marco-      │  │
│  │              │  │              │  │  MiniLM-L-6-v2)  │  │
│  └──────────────┘  └──────────────┘  └──────────────────┘  │
├─────────────────────────────────────────────────────────────┤
│  CAPA DE ALMACENAMIENTO                                    │
│  ┌──────────────────────────────────────────────────────┐   │
│  │  ChromaDB (HNSW + Cosine) — 700 documentos indexados │   │
│  └──────────────────────────────────────────────────────┘   │
├─────────────────────────────────────────────────────────────┤
│  CAPA DE GENERACIÓN                                        │
│  ┌──────────────────────────────────────────────────────┐   │
│  │  Gemini (LLM) + LangChain PromptTemplate            │   │
│  │  Contexto inyectado + Memoria conversacional         │   │
│  └──────────────────────────────────────────────────────┘   │
└─────────────────────────────────────────────────────────────┘
```

**Módulos del sistema (`src/`):**

| Módulo | Responsabilidad |
| :--- | :--- |
| `data_processing.py` | Descarga y preprocesa el corpus (texto + imágenes) |
| `embeddings.py` | Generación de embeddings multimodales con CLIP |
| `index_corpus.py` | Indexación de embeddings de imagen en ChromaDB |
| `vector_db.py` | Interfaz de persistencia y búsqueda vectorial (ChromaDB) |
| `retrieval.py` | Pipeline de recuperación (Bi-Encoder, Cross-Encoder, Query Expansion) |
| `query_expansion.py` | Expansión automática de consultas con Gemini |
| `generation.py` | Generación RAG con LangChain + memoria conversacional |
| `relevance_feedback.py` | Retroalimentación del usuario (Rocchio score-based) |
| `evaluation_metrics.py` | Implementación de Precision@k, Recall@k, NDCG@k |
| `evaluate.py` | Script de evaluación contra qrels |
| `app.py` | Interfaz web Streamlit (chat multimodal) |

## 3. Pipeline de Recuperación y Generación

### 3.1 Indexación Offline

1. Se cargan los 700 productos del corpus JSON.
2. Cada imagen pasa por el **Visual Encoder de CLIP (ViT-B-32)** para producir un vector de 512 dimensiones.
3. Los embeddings se normalizan con norma $L_2$ (magnitud unitaria).
4. Se insertan en **ChromaDB** (índice HNSW, métrica coseno) junto con el texto y metadatos asociados.

### 3.2 Consulta Online — Texto (Pipeline de 3 etapas)

**Stage 0 — Query Expansion (Gemini):**  
La consulta original se envía a Gemini, que genera 3 reformulaciones semánticas adicionales. Esto produce 4 variantes de búsqueda en total.

**Stage 1 — Bi-Encoder (CLIP):**  
Cada variante se vectoriza con el Text Encoder de CLIP y se buscan los top-15 candidatos en ChromaDB por similitud coseno. Se agrupan todos los candidatos y se eliminan duplicados por ID (~60 candidatos únicos máx.).

**Stage 2 — Cross-Encoder Re-ranking:**  
El modelo `ms-marco-MiniLM-L-6-v2` evalúa cada par (consulta original, texto del candidato) con atención cruzada completa, produciendo un score de relevancia semántica. Los candidatos se reordenan y se retornan los top-3 finales.

### 3.3 Consulta Online — Imagen

La imagen subida pasa por el Visual Encoder de CLIP para obtener su embedding y se busca directamente en ChromaDB. No se aplica re-ranking (el Cross-Encoder opera solo sobre pares texto-texto).

### 3.4 Consulta Online — Voz

El audio se transcribe a texto usando la API nativa de Gemini (`mime_type: audio/wav`) y luego se trata como una consulta de texto, pasando por el pipeline de 3 etapas.

### 3.5 Generación de Respuesta (RAG)

Una vez obtenidos los documentos relevantes:

1. Se formatean los productos recuperados como contexto textual estructurado.
2. Se inyecta el **historial conversacional** (últimos 6 turnos) como variable `{chat_history}`.
3. Se construye el prompt final con LangChain `PromptTemplate` y se envía a Gemini.
4. El LLM genera una respuesta contextualizada que referencia los productos encontrados.

## 4. Resultados Experimentales

### 4.1 Configuración del Experimento

- **Conjunto de evaluación (qrels):** 20 consultas anotadas manualmente, cubriendo 3 categorías (Musical Instruments, Pet Supplies, Video Games).
- **Documentos relevantes por consulta:** entre 1 y 5 (promedio ~2).
- **Pipeline evaluado:** Stage 1 únicamente (CLIP Bi-Encoder puro, sin re-ranking ni expansión), para medir la calidad base del sistema.
- **Valores de k evaluados:** 1, 3, 5, 10.

### 4.2 Resultados Globales (Promedios)

| Métrica | @1 | @3 | @5 | @10 |
| :--- | :---: | :---: | :---: | :---: |
| **Precision** | 0.6000 | 0.3167 | 0.2700 | 0.1550 |
| **Recall** | 0.3392 | 0.4883 | 0.6650 | 0.7750 |
| **NDCG** | 0.6000 | 0.5211 | 0.5937 | 0.6361 |

### 4.3 Ejemplos de Resultados por Consulta

| Query | P@3 | R@3 | NDCG@3 |
| :--- | :---: | :---: | :---: |
| drum set for beginners | 0.3333 | 1.0000 | 1.0000 |
| acoustic guitar | 0.6667 | 1.0000 | 1.0000 |
| dog harness adjustable no pull | 0.3333 | 0.2000 | 0.2471 |
| gaming headset with microphone | 0.3333 | 0.2500 | 0.3655 |
| zelda nintendo game | 0.0000 | 0.0000 | 0.0000 |
| PS5 stand with cooling fan | 0.0000 | 0.0000 | 0.0000 |

## 5. Análisis de las Métricas de Evaluación

### Precision@k

La **Precision@1 = 0.60** indica que en el 60% de las consultas, el producto más relevante aparece en la primera posición. La caída progresiva hacia P@10 = 0.155 es esperable: al aumentar k, se incluyen más documentos no relevantes en el denominador.

La baja Precision@3 (0.3167) revela que CLIP como Bi-Encoder es eficaz para encontrar al menos un resultado correcto, pero no es preciso al ordenar los primeros 3 resultados. Este comportamiento justificó la implementación del **Cross-Encoder** como etapa de re-ranking.

### Recall@k

El **Recall@10 = 0.775** demuestra que CLIP captura el 77.5% de los documentos relevantes dentro de los 10 primeros candidatos. Esto confirma que el Bi-Encoder funciona como un excelente motor de *recall*, trayendo la gran mayoría de los documentos relevantes a un pool reducido sobre el cual un re-ranker puede actuar.

El crecimiento de Recall@1 (0.34) → Recall@10 (0.775) evidencia que ampliar la ventana de recuperación es beneficioso, lo cual sustenta el diseño del pipeline de dos etapas.

### NDCG@k

El **NDCG@5 = 0.5937** y **NDCG@10 = 0.6361** muestran un ranking decente pero con margen de mejora. Dado que NDCG penaliza la posición en la que aparecen los documentos relevantes, un NDCG sub-óptimo indica que los resultados relevantes no siempre están en las primeras posiciones. Un Cross-Encoder que reordene el Top-20 al Top-3 debería incrementar significativamente esta métrica.

### Consultas con rendimiento cero

Las consultas `q16` ("zelda nintendo game") y `q20` ("PS5 stand with cooling fan") obtuvieron 0.0 en todas las métricas. Esto se debe a que los productos relevantes anotados en las qrels no fueron recuperados por CLIP dentro del Top-10. Posibles causas:
- El embedding textual no captura adecuadamente nombres propios de videojuegos específicos.
- La búsqueda se realiza sobre embeddings de **imagen** (indexación visual), por lo que queries muy textuales/nominales pueden no tener correspondencia visual directa.

Estas consultas se beneficiarían particularmente de la **Query Expansion**, que introduce variantes semánticas para cubrir vocabulario adicional.

## 6. Funcionalidades de Excelencia Implementadas

Se implementaron las 4 funcionalidades opcionales para alcanzar el máximo puntaje adicional (+60 pts).

### A. Re-ranking con Cross-Encoder (+15 pts)

**Modelo:** `cross-encoder/ms-marco-MiniLM-L-6-v2` (sentence-transformers).  
**Implementación:** Clase `MultimodalRetriever.retrieve_and_rerank()` en `src/retrieval.py`.

- Stage 1 (CLIP) recupera un pool de 20 candidatos.
- Stage 2 (Cross-Encoder) evalúa cada par (query, documento) con atención cruzada completa.
- Se reordenan los candidatos por score de relevancia semántica y se devuelven los top-3.
- El Cross-Encoder se carga de forma lazy para no consumir memoria hasta que sea necesario.

### B. Query Expansion con LLM (+15 pts)

**Modelo:** Gemini (API de Google).  
**Implementación:** Clase `QueryExpander` en `src/query_expansion.py`.

- Antes de la búsqueda, se envía la consulta a Gemini con un prompt que solicita N reformulaciones.
- Gemini retorna un JSON con las variantes (sinónimos, términos técnicos, traducciones).
- Se ejecutan búsquedas independientes en ChromaDB para cada variante.
- Los resultados se agrupan eliminando duplicados por ID antes del re-ranking.
- En caso de error con la API, se degrada gracefully usando solo la consulta original.

### C. Relevance Feedback — Rocchio Score-Based (+15 pts)

**Implementación:** Clase `RelevanceFeedbackStore` en `src/relevance_feedback.py`.

- La interfaz presenta botones 👍/👎 junto a cada producto recomendado.
- Cada voto se persiste en `data/evaluation/relevance_feedback.json`.
- Se calcula un factor de boost/penalización por documento:  
  `factor = 1.0 + α × (likes − dislikes) / (total + 1)` donde α = 0.3.
- El factor multiplica el score del Cross-Encoder (o CLIP) en búsquedas futuras.
- Rango efectivo del factor: ~0.7 (penalizado) a ~1.3 (boosteado).
- Se re-ordena la lista de resultados tras aplicar los factores.

### D. Memoria Conversacional (+15 pts)

**Implementación:** Método `RAGGenerator.format_chat_history()` en `src/generation.py`.

- Se mantiene el historial completo de la sesión en `st.session_state.messages`.
- Antes de cada generación, se extraen los últimos 6 mensajes (ventana deslizante).
- Los mensajes del asistente se truncan a 300 caracteres para no saturar la ventana de contexto.
- El historial se inyecta en el prompt RAG como variable `{chat_history}`.
- Esto permite al LLM resolver referencias anafóricas ("ese producto", "el primero", "algo más barato") y mantener coherencia conversacional entre turnos.